In [3]:
import sys
import matplotlib.pyplot as plt
plt.rc("axes.spines", top=False, right=False)

import seaborn as sns
import pandas as pd
import numpy as np
from scipy import stats

from tqdm import tqdm

###changes default of autoreloader to continually reload (2) rather than only on restart
####good for debuggin but does slow code as it is continully reloading modules
%load_ext autoreload
%autoreload 2

###can also use dir() to check what is in function and that new params have been loaded

import os
import glob


#desktop = os.path.join(os.path.join(os.environ['USERPROFILE']), 'Desktop') 
plot_path="../output/plots"
mypath = plot_path
import sys
sys.path.append("../src")
from Branch_analysis import branch_dsc, update_DSCs, update_compartments
from Summary_analysis import summary_dsc, summary_compt, avgCDF
from plots import plot_dsc

recap the figs 
plot individual dendrites
summary values
    mean diam
    length
    distance
    resistance


## Load dendritic diameter data
Set location of repo data (root folder) sets the paths for each expanded dendrite analysed

In [6]:
from pathlib import Path
root='M:/Tony/Bonn/Data/Distal_Dendrite/Morphology/'

DGdata={}

DGdata['dendrite1'] = {'path':root+'repo_data/ExM/2016_07_07/branch1/tables/2016_07_07_slice4_left_branch1_Local Diameter.csv'}
DGdata['dendrite1']['file']= '2016_07_07_slice4_left_branch1_Local Diameter.csv'
DGdata['dendrite1']['ex_factor'] = 4.1
DGdata['dendrite1']['res'] = 1

#sub-branch of branch 3 
# DGdata['dendrite2'] = {'path':''M:/Tony/Bonn/Data/Distal_Dendrite/Morphology/repo_data/ExM/2016_07_07/branch2/tables/2016_07_07_slice4_left_branch2_Local Diameter.csv'}
# DGdata['dendrite2']['file']= '2016_07_07_slice4_left_branch2_Local Diameter.csv'
# DGdata['dendrite2']['ex_factor'] = 4.1
# DGdata['demdrite2']['res'] = 1

DGdata['dendrite3'] = {'path':root+'repo_data/ExM/2016_07_07/branch3/tables/2016_07_07_slice4_left_branch3_Local Diameter.csv'}
DGdata['dendrite3']['file']= '2016_07_07_slice4_left_branch3_Local Diameter.csv'
DGdata['dendrite3']['ex_factor'] = 4.1
DGdata['dendrite3']['res'] = 1

DGdata['dendrite4'] = {'path':root+'repo_data/ExM/2019_05_04/branch4/tables/2019_05_04_slice3_side1_branch4_localDiameter.csv'}
DGdata['dendrite4']['file']= '2019_05_04_slice3_side1_branch4_localDiameter.csv'
DGdata['dendrite4']['ex_factor'] = 4.1
DGdata['dendrite4']['res'] = 1

DGdata['C1_02_cell1_d2'] = {'path':root+'repo_data/ExM/2020_06_08/T1_02/tables/20210201_DG-T1_AF488-02_neuron01_8bit_1000-3000_dend2 - Local Diameter.csv'}
DGdata['C1_02_cell1_d2']['file']= '20210201_DG-T1_AF488-02_neuron01_8bit_1000-3000_dend2 - Local Diameter.csv'
DGdata['C1_02_cell1_d2']['ex_factor'] = 3.8
DGdata['C1_02_cell1_d2']['res'] = 0.173

DGdata['C1_03_cell2_d2'] = {'path':root+'repo_data/ExM/2020_06_08/T1_03/tables/20210201_dg-t1_af488-03_decon-Resolution Level 1_8bit_480-2600_dend2 - Local Diameter.csv'}
DGdata['C1_03_cell2_d2']['file']= '20210201_dg-t1_af488-03_decon-Resolution Level 1_8bit_480-2600_dend2 - Local Diameter.csv'
DGdata['C1_03_cell2_d2']['ex_factor'] = 3.8
DGdata['C1_03_cell2_d2']['res'] = 0.173

DGdata['C1_03_cell2_d3']={'path' : root+'/repo_data/ExM/2020_06_08/T1_03/tables/20210201_dg-t1_af488-03_decon-Resolution Level 1_8bit_480-2600_dend3 - Local Diameter.csv'}
DGdata['C1_03_cell2_d3']['file']= '20210201_dg-t1_af488-03_decon-Resolution Level 1_8bit_480-2600_dend3 - Local Diameter.csv'
DGdata['C1_03_cell2_d3']['ex_factor'] = 3.8
DGdata['C1_03_cell2_d3']['res'] = 0.173

In [7]:
dataSet=DGdata
for name, dendrite in tqdm(dataSet.items()):
    #print(name)
    #print(data[name]['path'])
    
    file_path = dendrite['path']#+dendrite['file']
    dataSet[name] = branch_dsc(file_path, dataSet[name]['res'], dataSet[name]['ex_factor'])

100%|██████████| 6/6 [00:00<00:00, 10.66it/s]


manually adjust DSC
    - add DSCs missing as diam increses slighlty in the middle
    - remove false postives

In [4]:
dendrite = DGdata['C1_03_cell2_d3']
DSC=dendrite['data'][3947:4047].copy() # makes a copy to work on
DSC.diam*=1/dendrite['ex_factor']   #if the DSCs overlap each other the scaling applied again. Separate DSC from orig data
DSC.smth*=1/dendrite['ex_factor']
DSC.length*=dendrite['res']/dendrite['ex_factor']
DSC.pt_distance*=dendrite['res']/dendrite['ex_factor']
dendrite['DSCs'].append(DSC) # add the changes back to the copy of the original data

update_DSCs(dendrite)
update_compartments(dendrite)

dendrite = DGdata['C1_03_cell2_d2']
del dendrite['DSCs'][2] #the auto detected DSC is too long past the DSC -shortened

update_DSCs(dendrite)
update_compartments(dendrite)


dendrite = DGdata['dendrite1']
del dendrite['DSCs'][11] #noisey false positives
del dendrite['DSCs'][10] #noisey false positives
del dendrite['DSCs'][9] #noisey false positives
del dendrite['DSCs'][8] #noisey false positives
del dendrite['DSCs'][7] #
del dendrite['DSCs'][2] #false positive

update_DSCs(dendrite)
update_compartments(dendrite)

In [17]:
from Branch_analysis import add_branch, FindBranch
add_branch(DGdata)
FindBranch(DGdata)


### Create figure panels for Fig1 
Fig1g DSC mean diameter
Fig1h DSC lenght
Fig1j DSC distance
select paramtert to plot [mean, lenght, distance]

In [ ]:
#select property to plot [mean, lenght, distance]
parameter='mean'
df= summary_dsc(DGdata, parameter)

# plot the cumulative histogram for each dendrite
n_bins=200
n, bins, patches = plt.hist(df.transpose(), n_bins, density=True, histtype='step', cumulative=True)
# Close the plot to suppress it
plt.close()

avg_df, bins = avgCDF(df, patches, n_bins, n)

fig, ax = plt.subplots(figsize=(4, 4))
fig.suptitle("DSC_"+parameter)
ax.plot(avg_df.iloc[:, :-1], bins, color='grey')
ax.plot(avg_df.iloc[:, len(avg_df.columns)-1], bins, color='black')

#ax.set_xlim([100,500])
print(f"{parameter} mean = {df.transpose().mean().mean()} +- sem {df.transpose().mean().sem()} count {df.melt().count().value}")


### Create figure panels for Fig. 1 
Fig. 1j Compartment length

In [ ]:
df= summary_compt(DGdata, 'length')

# plot the cumulative histogram for each dendrite
n_bins=200
n, bins, patches = plt.hist(df.transpose(), n_bins, density=True, histtype='step', cumulative=True)
# Close the plot to suppress it
plt.close()

avg_df, bins = avgCDF(df, patches, n_bins, n)

fig, ax = plt.subplots(figsize=(4, 4))
fig.suptitle("Compartment_length")
ax.plot(avg_df.iloc[:, :-1], bins, color='grey')
ax.plot(avg_df.iloc[:, len(avg_df.columns)-1], bins, color='black')

ax.set_xlim([0,50])
print(f"{parameter} mean = {df.transpose().mean().mean()} +- sem {df.transpose().mean().sem()} count {df.melt().count().value}")


### Select dendrite and branch to analyse
Example dendrite shown in Supplemental Fig. 1b 

In [22]:
#data, dendrite branch
dend_name= 'C1_03_cell2_d3'  #'dendrite3' #'dendrite1' 'dendrite4' #'C1_02_cell1_d2'   #'C1_03_cell2_d2' 'C1_03_cell2_d3'
branchNo= 3
#branchs=dendbranch_dic[dend_name]
df=DGdata[dend_name]['DSC_properties']

In [ ]:
fig, ax = plot_dsc(DGdata[dend_name],dend_name, corr=1)  

In [8]:
#sets start and end points of branch
strt = DGdata[dend_name]['branchNo'][branchNo][1]

if branchNo + 1 < len(DGdata[dend_name]['branchNo']) and DGdata[dend_name]['branchNo'][branchNo+1] in DGdata[dend_name]['branchNo']:
    end = DGdata[dend_name]['branchNo'][branchNo+1][1]
else:
    end = (DGdata[dend_name]['data']['length'].iloc[-1] * 
           DGdata[dend_name]['res'] / 
           DGdata[dend_name]['ex_factor'])

ax.set_xlim(strt, end)
fig

NameError: name 'dend_name' is not defined

### Create figure panels for supplemental Fig. 1
Example dendrite shown in Supplemental Fig. 1b 

In [ ]:
from plots import branch_figs
branch_figs(DGdata, plot_path)

### Calculate Resistances for DSCs and dendritic shafts
Resistance values estimated from morphology (see [Tonnesen et al. 2014](...]) ) using 
Rmorph = (p x L) / A
p= resistivity of 181 Ohms cm
L = length 
A = cross sectional area

In [25]:
from Branch_analysis import resistances

dendbranch_dic={'dendrite1': [1,2],'dendrite3':[1,2,3], 'dendrite4':[2], 'C1_02_cell1_d2':[3], 'C1_03_cell2_d2':[5],'C1_03_cell2_d3':[3]}

for dend, branchs in dendbranch_dic.items():
    #print(dend)
    for b in branchs:
        #print(b)
        resistances(dend, b, DGdata)

In [26]:
#compile all resistance data 
Rmorph_data={}
length_data={}
Rdend_data={}
#Rmorph_data['dendrite3'] = []  # Initialize a list for each new key

for dendrite, d in DGdata.items():
    #print(dendrite)
    for b in d['DSC_properties']:
        #print(b)
        if 'Rmorph' in b:
            #print(dsc)
            #print(b['Rmorph'])
            if dendrite not in Rmorph_data:
                Rmorph_data[dendrite] = []  # Initialize a list for each new key
                length_data[dendrite] = []
            Rmorph_data[dendrite].append(b['Rmorph'])
            length_data[dendrite].append(b['length'])
    if 'Rdend' in d['branch_info']:
        Rdend_data[dendrite]=pd.DataFrame(d['branch_info']['Rdend'], columns=['resistance'])
# DSC data for the DataFrame
df_data = [(key, value) for key, values in Rmorph_data.items() for value in values]
length_data=[(key, value) for key, values in length_data.items() for value in values]

In [ ]:
df = pd.concat(Rdend_data)
df = df.reset_index(level=0)
df = df.rename(columns={'level_0': 'dendrite'})

grp_df = df.groupby('dendrite').mean(numeric_only=True)

fig, ax =plt.subplots(figsize=(4, 4))
ax=sns.stripplot(y='resistance', alpha=0.4, data=grp_df)
ax=sns.pointplot(y='resistance', linestyle='none', errorbar='se',data=grp_df)

print(f"Dend resisitance per micron {grp_df.mean(numeric_only=True)} SEM {grp_df.sem(numeric_only=True)} count {grp_df.count()}")

# Create the DataFrame
df = pd.DataFrame(df_data, columns=['dendrite', 'resistance'])
length_df = pd.DataFrame(length_data, columns=['dendrite', 'length'])

ax=sns.stripplot(y='resistance', alpha=0.4, data=df)
ax=sns.pointplot(y='resistance', linestyle='none', errorbar='se',data=df)

print(f"DSC resisitance per mircon {df.mean(numeric_only=True)} SEM {df.sem(numeric_only=True)} count {df.count()}")

ResisTotal={'resitance':length_df.length*df.resistance, 'dendrite': df.dendrite}
df=pd.DataFrame(ResisTotal)

grp_df = df.groupby('dendrite').sum(numeric_only=True)
print(f"Sum DSC resisitance per dendrite {grp_df.mean(numeric_only=True)} SEM {grp_df.sem(numeric_only=True)} count {grp_df.count()}")
